# Contextual Embeddings

This notebook examines how a pretrained transformer represents the expressions *frankenstein* and *frankensteining*.

Building on the static embedding analysis in Notebook 3, we investigate subword tokenization and contextual representations using DistilBERT.

The objective is to explore how representations vary across linguistic contexts, rather than to evaluate whether the model understands the
intended meaning of either expression.

### Comparison design

The analysis proceeds from constructed examples to authentic conversational occurrences, examining how the contextual representations of *frankenstein* and *frankensteining* vary across grammatical and discourse environments.

**1. Constructed examples**

Two sentences establish an initial comparison between an infinitival transitive verb and a nominalized *-ing* form:

- **Verbal:** *We decided to Frankenstein the experimental dataset.*
- **Nominal:** *We wrote a guide explaining the frankensteining of the experimental dataset.*

The examples were designed to approximate grammatical constructions observed in the corpus while providing relatively self-contained linguistic contexts. Unlike the authentic occurrences, they do not depend on the conversational history in which the expressions originated.

**2. User-produced corpus examples**

The analysis then examines two authentic occurrences:

- **EX001:** *decided to Frankenstein it* — the original verbal innovation.
- **EX004:** *explaining the frankensteining and the unicorning* — a subsequent nominal use.

These occurrences preserve the original linguistic environments in which the expressions were used. Their shared conceptual reference is established by the conversational history, which is not supplied to DistilBERT when the excerpts are processed independently.

**3. Assistant-produced corpus example**

Finally, the analysis introduces an assistant-produced nominal occurrence:

- **EX010:** *Faculty of Counterfactual Frankensteining and Applied Gibberishonian Studies.*

This occurrence was selected because frankensteining functions as the head of a nominal expression rather than as a premodifier of another noun. It also represents an unsolicited creative extension of the user-introduced lexical innovation.

The resulting comparisons examine variation across constructed and authentic contexts, grammatical constructions, and conversational uses. Cosine similarity measures relationships between contextual representations, but does not independently establish semantic interpretation or recognition of the expressions' shared conversational history.

## 1. Model and tokenizer

In [1]:
# Load tokenizer and pre-trained model

from transformers import AutoTokenizer

model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

from transformers import AutoModel

model = AutoModel.from_pretrained(model_name)
model.eval()

/opt/anaconda3/envs/novel-language-nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 18:15:59.926000 82985 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8206.26it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arc

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [2]:
# Inspect subword tokenization

target_words = ["frankenstein", "frankensteining"]

tokens = {word: tokenizer.tokenize(word) for word in target_words}
print(tokens)

{'frankenstein': ['frankenstein'], 'frankensteining': ['frankenstein', '##ing']}


### Subword tokenization

DistilBERT's WordPiece tokenizer represents *frankenstein* as a single token (`frankenstein`), while *frankensteining* is segmented into two subword tokens (`frankenstein` and `##ing`).

This allows the novel *-ing* expression to be processed without a dedicated whole-word vocabulary entry.

The distinction is relevant to the subsequent comparisons: the contextual representation of *frankenstein* is extracted from one token, whereas that of *frankensteining* is obtained by averaging the representations of its two subword tokens.

## 2. Constructed Examples

Unlike static word embeddings, transformer representations are influenced
by the surrounding linguistic context.

This section extracts contextual embeddings for our target expressions
using a pretrained DistilBERT model.

We first examine the model's output structure and embedding dimensions
before comparing representations across different contexts.

### 2.1. Grammatical context: verbal vs. nominal constructions

The constructed examples provide an initial comparison between two grammatical realizations of the same lexical innovation: *Frankenstein* as an infinitival transitive verb and *frankensteining* as a nominalized *-ing* form.

Both sentences refer to the experimental dataset and approximate constructions observed in the conversational corpus:

- **Verbal:** *We decided to Frankenstein the experimental dataset.*
- **Nominal:** *We wrote a guide explaining the frankensteining of the experimental dataset.*

We examine how DistilBERT tokenizes these expressions in context and extract their contextual embeddings for comparison.

Although the examples share a referent, they differ in morphology, grammatical function and surrounding vocabulary. Consequently, differences between their representations cannot be attributed exclusively to grammatical function.

In [3]:
import torch

sentences = {
    "verbal": "We decided to Frankenstein the experimental dataset.",
    "nominal": "We wrote a guide explaining the frankensteining of the experimental dataset."
}

for construction, sentence in sentences.items():

    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    print(f"\n{construction.upper()}")
    print("Input tokens:", tokens)
    print("Embedding shape:", outputs.last_hidden_state.shape)


VERBAL
Input tokens: ['[CLS]', 'we', 'decided', 'to', 'frankenstein', 'the', 'experimental', 'data', '##set', '.', '[SEP]']
Embedding shape: torch.Size([1, 11, 768])

NOMINAL
Input tokens: ['[CLS]', 'we', 'wrote', 'a', 'guide', 'explaining', 'the', 'frankenstein', '##ing', 'of', 'the', 'experimental', 'data', '##set', '.', '[SEP]']
Embedding shape: torch.Size([1, 16, 768])


In [4]:
# Extract contextual embeddings

embeddings = {}

for construction, sentence in sentences.items():
    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Locate the target expression
    start = tokens.index("frankenstein")

    # Account for the different numbers of subword tokens
    n_tokens = 1 if construction == "verbal" else 2

    # Extract the target representation
    embedding = outputs.last_hidden_state[
        0, start:start + n_tokens
    ].mean(dim=0)

    embeddings[construction] = embedding

    print(construction, "| Embedding shape:", embedding.shape)

verbal | Embedding shape: torch.Size([768])
nominal | Embedding shape: torch.Size([768])


In [5]:
# Calculate cosine similarity

import torch.nn.functional as F

similarity = F.cosine_similarity(
    embeddings["verbal"].unsqueeze(0),
    embeddings["nominal"].unsqueeze(0)
)

print(f"Cosine similarity: {similarity.item():.4f}")

Cosine similarity: 0.7008


### 2.2. Findings

DistilBERT tokenized the verbal form *Frankenstein* as a single token (`frankenstein`) and the nominal form *frankensteining* as two subword tokens (`frankenstein`, `##ing`).

The contextual representation of each target expression was extracted using its corresponding token span. For the nominal form, the two subword representations were averaged to obtain a single 768-dimensional vector.

The resulting embeddings yielded a **cosine similarity of 0.7008**, indicating positive directional similarity between the representations.

Although both expressions refer to a related activity, their linguistic environments differ in morphology, grammatical function and surrounding vocabulary. The observed similarity therefore cannot be attributed to any individual linguistic factor.

The result provides an initial reference point for examining the corresponding expressions in authentic conversational contexts.

## 3. User-produced corpus examples

Having examined the target expressions in constructed linguistic contexts, we now turn to their authentic occurrences in the conversational corpus.

Unlike the constructed examples, these expressions were produced within an ongoing interaction in which their intended meanings and references developed through shared discourse.

The analysis examines how DistilBERT represents these occurrences when they are processed independently, without access to the preceding conversational history.

### 3.1. Authentic verbal and nominal constructions

Two user-produced occurrences were selected from the linguistic annotations:

- **EX001:** *decided to Frankenstein it* — the original innovative use of *Frankenstein* as a transitive verb.
- **EX004:** *explaining the frankensteining and the unicorning* — a subsequent nominal use of *frankensteining*.

The original contexts are retrieved from `linguistic_annotations.csv`. Shorter excerpts are then selected for embedding extraction, preserving the relevant grammatical constructions and surrounding linguistic context.

EX001 begins mid-utterance, as recorded in the annotations. Its excerpt retains the preceding references to the dataset and the subsequent evaluation of the activity, while excluding unrelated conversational material.

The same DistilBERT model and extraction procedure used for the constructed examples are applied here, allowing the resulting contextual representations to be compared.

In [6]:
import pandas as pd

corpus_df = pd.read_csv("../data/public/linguistic_annotations.csv")

selected = corpus_df[
    (corpus_df["exchange_id"].isin(["EX001", "EX004"])) &
    (corpus_df["speaker"] == "user")
]

for _, row in selected.iterrows():
    print(f"\n{row['exchange_id']}")
    print("LEFT:", row["left_context"])
    print("FORM:", row["form"])
    print("RIGHT:", row["right_context"])


EX001
LEFT: actually pruned a lot of it, made Russian dolls with it, but then we decided to 
FORM: Frankenstein
RIGHT:  it. And it worked, and it was epic!

OK, I need lunch and a shower, amigo. But 

EX004
LEFT:  I need to change the zombie roadmap that turned into a guide to explaining the 
FORM: frankensteining
RIGHT:  and the unicorning, because of the finalísimo's numbering before the appendices


In [7]:
# Define user_excerpts to be used and tokenize

user_excerpts = {
    "EX001": (
        "actually pruned a lot of it, made Russian dolls with it, "
        "but then we decided to Frankenstein it. "
        "And it worked, and it was epic!"
    ),
    "EX004": (
        "I need to change the zombie roadmap that turned into a guide "
        "to explaining the frankensteining and the unicorning, "
        "because of the finalísimo's numbering before the appendices."
    )
}

for exchange_id, sentence in user_excerpts.items():
    print(f"\n{exchange_id}")
    print(tokenizer.tokenize(sentence))


EX001
['actually', 'pr', '##une', '##d', 'a', 'lot', 'of', 'it', ',', 'made', 'russian', 'dolls', 'with', 'it', ',', 'but', 'then', 'we', 'decided', 'to', 'frankenstein', 'it', '.', 'and', 'it', 'worked', ',', 'and', 'it', 'was', 'epic', '!']

EX004
['i', 'need', 'to', 'change', 'the', 'zombie', 'road', '##ma', '##p', 'that', 'turned', 'into', 'a', 'guide', 'to', 'explaining', 'the', 'frankenstein', '##ing', 'and', 'the', 'unicorn', '##ing', ',', 'because', 'of', 'the', 'final', '##isi', '##mo', "'", 's', 'numbering', 'before', 'the', 'app', '##end', '##ices', '.']


In [8]:
# Extract contextual embeddings

user_embeddings = {}

for exchange_id, sentence in user_excerpts.items():
    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Locate the target expression
    start = tokens.index("frankenstein")

    # EX001 has one token; EX004 has two
    n_tokens = 1 if exchange_id == "EX001" else 2

    embedding = outputs.last_hidden_state[
        0, start:start + n_tokens
    ].mean(dim=0)

    user_embeddings[exchange_id] = embedding

    print(exchange_id, "| Target tokens:", tokens[start:start + n_tokens])
    print("Embedding shape:", embedding.shape)

EX001 | Target tokens: ['frankenstein']
Embedding shape: torch.Size([768])
EX004 | Target tokens: ['frankenstein', '##ing']
Embedding shape: torch.Size([768])


In [9]:
# Cosine similarity

user_similarity = F.cosine_similarity(
    user_embeddings["EX001"].unsqueeze(0),
    user_embeddings["EX004"].unsqueeze(0)
)

print(f"User-produced expressions — cosine similarity: {user_similarity.item():.4f}")

User-produced expressions — cosine similarity: 0.5794


### 3.2. Findings

The contextual representations of the original verbal innovation (EX001: *Frankenstein*) and the subsequent nominal use (EX004: *frankensteining*) yielded a cosine similarity of **0.5794**.

This is lower than the similarity observed between the corresponding constructed examples (**0.7008**), which were designed to approximate the grammatical constructions and semantic contexts of the authentic occurrences.

Both comparisons involve the same morphological contrast and token-extraction procedure. However, the authentic excerpts contain additional lexical, syntactic and discourse variation that may contribute to the difference in similarity.

The result shows that the two user-produced expressions have less closely aligned contextual representations than their constructed counterparts. It does not establish which linguistic factors account for this difference or whether the model recognizes their shared conceptual origin.

## 4. Assistant-produced corpus example

Having examined the constructed examples and the original user-produced occurrences, we now introduce an assistant-produced use of *frankensteining*.

This occurrence extends the user-introduced lexical innovation into a new discourse context. Unlike the previous examples, it refers not directly to the reconstruction of the experimental dataset, but to an imaginary academic discipline.

The objective is to examine how DistilBERT represents this creative extension and compare its contextual embedding with those obtained from the constructed and user-produced examples.

### 4.1. Creative nominal extension

The selected occurrence is EX010:

*Dean Mari O, presiding over the newly established Faculty of Counterfactual Frankensteining and Applied Gibberishonian Studies.*

In this construction, *Frankensteining* functions as the head of a nominal expression modified by *Counterfactual*, rather than as a premodifier of another noun. This makes it grammatically comparable to the user's nominal use in EX004, although their syntactic environments and discourse functions differ.

The Faculty's name was generated by the assistant without an explicit request for that particular expression, drawing on lexical material previously introduced in the conversation.

We retrieve the occurrence from the linguistic annotations, select the relevant excerpt and examine its tokenization before extracting the contextual embedding.

The resulting representation will be compared with the user's original verbal innovation (EX001), the subsequent nominal use (EX004), and the constructed nominal example.

In [10]:
assistant_selected = corpus_df[
    (corpus_df["exchange_id"] == "EX010") &
    (corpus_df["speaker"] == "assistant")
]

for _, row in assistant_selected.iterrows():
    print("EXCHANGE:", row["exchange_id"])
    print("LEFT:", row["left_context"])
    print("FORM:", row["form"])
    print("RIGHT:", row["right_context"])
    print("CONSTRUCTION:", row["construction"])
    print("SYNTACTIC FUNCTION:", row["syntactic_function"])

EXCHANGE: EX010
LEFT: t.

Dean Mari O, presiding over the newly established Faculty of Counterfactual 
FORM: Frankensteining
RIGHT:  and Applied Gibberishonian Studies. 😂

Tomorrow: EL FINALÍSIMO. 🦄🌈🎤

Tonight: a
CONSTRUCTION: Faculty of Counterfactual Frankensteining and Applied Gibberishonian Studies
SYNTACTIC FUNCTION: Nominalised -ing head modified by 'Counterfactual'; part of a coordinated noun phrase functioning as the complement of the preposition 'of'


In [11]:
assistant_excerpt = (
    "Dean Mari O, presiding over the newly established Faculty "
    "of Counterfactual Frankensteining and Applied Gibberishonian Studies."
)

print(tokenizer.tokenize(assistant_excerpt))


['dean', 'mari', 'o', ',', 'presiding', 'over', 'the', 'newly', 'established', 'faculty', 'of', 'counter', '##fa', '##ct', '##ual', 'frankenstein', '##ing', 'and', 'applied', 'gi', '##bber', '##ish', '##onia', '##n', 'studies', '.']


In [12]:
# Extract contextual embedding

inputs = tokenizer(assistant_excerpt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Locate the target expression
start = tokens.index("frankenstein")

# Frankensteining occupies two subword tokens
n_tokens = 2

# Average the contextual representations
assistant_embedding = outputs.last_hidden_state[
    0, start:start + n_tokens
].mean(dim=0)

print("EX010 | Target tokens:", tokens[start:start + n_tokens])
print("Embedding shape:", assistant_embedding.shape)

EX010 | Target tokens: ['frankenstein', '##ing']
Embedding shape: torch.Size([768])


In [13]:
comparisons = {
    "User verbal (EX001)": user_embeddings["EX001"],
    "User nominal (EX004)": user_embeddings["EX004"],
    "Constructed nominal": embeddings["nominal"]
}

for label, reference_embedding in comparisons.items():
    similarity = F.cosine_similarity(
        assistant_embedding.unsqueeze(0),
        reference_embedding.unsqueeze(0)
    )

    print(f"EX010 vs. {label}: {similarity.item():.4f}")

EX010 vs. User verbal (EX001): 0.5221
EX010 vs. User nominal (EX004): 0.7860
EX010 vs. Constructed nominal: 0.7633


### 4.2. Findings

The assistant-produced occurrence (EX010) yielded the following cosine similarities with the previously extracted contextual embeddings:

| Comparison | Cosine similarity |
|---|---:|
| EX010 vs. user verbal (EX001) | 0.5221 |
| EX010 vs. user nominal (EX004) | 0.7860 |
| EX010 vs. constructed nominal | 0.7633 |

The assistant's nominal use of *Frankensteining* is represented more similarly to the user's nominal occurrence (EX004) than to the original verbal innovation (EX001). Its similarity to the constructed nominal example is also relatively high.

This pattern is consistent with closer representational alignment between occurrences sharing the same morphological form and nominal grammatical function, despite differences in their surrounding vocabulary, syntactic environments and discourse contexts.

However, the comparisons do not isolate the effects of morphology, grammatical function or lexical context. Nor do they demonstrate that the model recognises the conversational development of the innovation, since each excerpt was processed independently.

The results therefore indicate differences in contextual representation that are compatible with the observed linguistic distinctions, without establishing which factors account for those differences.